# Assess How Well Top-Ranked Ligands Can Predict a Gene Set of Interest

This notebook assesses the ligands prioritized by NicheNet in their ability to predict a gene set of interest. We first run a NicheNet analysis to obtain ligand rankings, then evaluate the top 30 ligands via cross-validated random forest classification, fraction-of-top-predicted analysis, and Fisher's exact test.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from functools import reduce
import seaborn as sns
sns.set_style('ticks')

import nichenetr as nn

## Run NicheNet

We run the NicheNet wrapper to obtain ligand rankings.

In [2]:
lr_network = nn.load_lr_network("mouse")
ligand_target_matrix = nn.load_ligand_target_matrix("mouse")
weighted_networks = nn.load_weighted_networks("mouse")

lr_network = lr_network[["from", "to"]].drop_duplicates()

adata = nn.load_seurat_obj()
adata = nn.alias_to_symbol_anndata(adata, "mouse")

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [3]:
nichenet_output = nn.nichenet_seuratobj_aggregate(
    receiver="CD8 T",
    adata=adata,
    sender=["CD4 T", "Treg", "Mono", "NK", "B", "DC"],
    condition_col="aggregate",
    condition_oi="LCMV",
    condition_ref="SS",
    celltype_col="celltype",
    expression_pct=0.05,
    ligand_target_matrix=ligand_target_matrix,
    lr_network=lr_network,
    weighted_networks=weighted_networks,
)

best_upstream_ligands = (
    nichenet_output["ligand_activities"]
    .nlargest(30, "aupr_corrected")["test_ligand"]
    .tolist()
)
print(f"Top 30 ligands: {best_upstream_ligands}")

Define expressed ligands and receptors in receiver and sender cells
Perform DE analysis in receiver cell
Perform NicheNet ligand activity analysis
Infer active target genes of the prioritized ligands
Infer receptors of the prioritized ligands
Perform DE analysis in sender cells
Top 30 ligands: ['Il27', 'Ebi3', 'Tnf', 'Ptprc', 'Vsig10', 'H2-Eb1', 'H2-M3', 'Clcf1', 'H2-T24', 'H2-T10', 'H2-T22', 'H2-T-ps', 'H2-M2', 'H2-T23', 'H2-Oa', 'App', 'H2-K1', 'H2-Q6', 'H2-Q4', 'H2-Q7', 'H2-D1', 'Siglech', 'Sirpa', 'Il18bp', 'Siglec1', 'Ccl5', 'Ccl22', 'Cd48', 'Cd320', 'Selplg']


## Assess how well top-ranked ligands can predict a gene set of interest

For the top 30 ligands, we build a multi-ligand model using cross-validated random forest classification. The model predicts whether a gene belongs to the gene set of interest.

In [4]:
# Cross-validation settings: 3-fold, 2 rounds
k = 3
n = 2

gene_predictions_top30_list = [
    nn.assess_rf_class_probabilities(
        round_num=i,
        folds=k,
        geneset=nichenet_output["geneset_oi"],
        background_expressed_genes=nichenet_output["background_expressed_genes"],
        ligands_oi=best_upstream_ligands,
        ligand_target_matrix=ligand_target_matrix,
    )
    for i in range(1, n + 1)
]

### Evaluate classification performance

Compute AUROC, AUPR, and Pearson correlation for each cross-validation round.

In [5]:
target_prediction_performances_cv = pd.concat(
    [
        nn.classification_evaluation_continuous_pred_wrapper(df).assign(round=i + 1)
        for i, df in enumerate(gene_predictions_top30_list)
    ],
    ignore_index=True,
)
target_prediction_performances_cv

,auroc,aupr,aupr_corrected,pearson,pearson_log_pval,spearman,spearman_log_pval,round
0,0.759618,0.414903,0.344132,0.477752,196.998204,0.230638,42.468105,1
1,0.759087,0.442995,0.372224,0.491996,210.635149,0.230167,42.294546,2


In [6]:
print(f"Mean AUROC: {target_prediction_performances_cv['auroc'].mean():.4f}")
print(f"Mean AUPR: {target_prediction_performances_cv['aupr'].mean():.4f}")
print(f"Mean Pearson: {target_prediction_performances_cv['pearson'].mean():.4f}")

Mean AUROC: 0.7594
Mean AUPR: 0.4289
Mean Pearson: 0.4849


### Fraction of top-predicted targets

Evaluate whether genes in the gene set are more likely to be among the top 5% predicted targets.

In [7]:
target_prediction_performances_discrete_cv = pd.concat(
    [
        nn.calculate_fraction_top_predicted(
            round_num=i + 1,
            response_prediction_df=df,
            ligands_oi=best_upstream_ligands,
            ligand_target_matrix=ligand_target_matrix,
            quantile_cutoff=0.95,
        ).assign(round=i + 1)
        for i, df in enumerate(gene_predictions_top30_list)
    ],
    ignore_index=True,
)
target_prediction_performances_discrete_cv

,true_target,n,positive_prediction,fraction_positive_predicted,round
0,False,3230,71,0.021981,1
1,True,246,103,0.418699,1
2,False,3230,70,0.021672,2
3,True,246,104,0.422764,2


In [8]:
# Fraction of true targets among top 5% predicted
frac_true = (
    target_prediction_performances_discrete_cv[
        target_prediction_performances_discrete_cv["true_target"] == True
    ]["fraction_positive_predicted"].mean()
)
print(f"Fraction of gene-set genes in top 5% predicted: {frac_true:.4f}")

frac_false = (
    target_prediction_performances_discrete_cv[
        target_prediction_performances_discrete_cv["true_target"] == False
    ]["fraction_positive_predicted"].mean()
)
print(f"Fraction of non-gene-set genes in top 5% predicted: {frac_false:.4f}")

Fraction of gene-set genes in top 5% predicted: 0.4207
Fraction of non-gene-set genes in top 5% predicted: 0.0218


### Fisher's exact test

Test whether gene-set members are significantly enriched in the top-predicted targets.

In [9]:
fisher_pvalues = [
    nn.calculate_fraction_top_predicted_fisher(
        round_num=i + 1,
        response_prediction_df=df,
        ligands_oi=best_upstream_ligands,
        ligand_target_matrix=ligand_target_matrix,
        quantile_cutoff=0.95,
    )
    for i, df in enumerate(gene_predictions_top30_list)
]

print(f"Fisher p-values: {fisher_pvalues}")
print(f"Average Fisher p-value: {np.mean(fisher_pvalues):.6f}")

Fisher p-values: [np.float64(6.605884681403625e-81), np.float64(2.0388352920851327e-82)]
Average Fisher p-value: 0.000000


### Get top-predicted genes

Look at which genes are well-predicted in every cross-validation round.

In [10]:
top_predicted_genes_list = [
    nn.get_top_predicted_genes(
        round_num=i + 1,
        response_prediction_df=gene_predictions_top30_list[i],
        ligands_oi=best_upstream_ligands,
        ligand_target_matrix=ligand_target_matrix,
    )
    for i in range(n)
]

# Combine across rounds
top_predicted_genes = reduce(
    lambda left, right: pd.merge(left, right, on=["gene", "true_target"], how="outer"),
    top_predicted_genes_list,
)

print("Top predicted true target genes:")
top_predicted_genes[top_predicted_genes["true_target"] == True]

Top predicted true target genes:


,gene,true_target,predicted_top_target_round1,predicted_top_target_round2
2,2410006H16Rik,True,True,NaN
4,Adar,True,True,True
6,Apobec3,True,True,True
10,B2m,True,True,True
11,Bst2,True,True,True
...,...,...,...,...
196,Ube2l6,True,True,True
197,Usp18,True,True,NaN
199,Xaf1,True,True,True
200,Zbp1,True,True,NaN
